Налаштування середовища та імпорт бібліотек
Ми активуємо режим %matplotlib qt, який створює окреме вікно для графіків.

In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, CheckButtons
from scipy import signal

Ініціалізація часу та шуму

In [3]:
t = np.linspace(0, 10, 1000)
current_noise = np.random.normal(0, np.sqrt(0.1), len(t))
last_noise_params = [0.0, 0.1] # mean, cov

Функція harmonic_with_noise (згідно з ТЗ)

In [4]:
def harmonic_with_noise(amplitude, frequency, phase, noise_mean, noise_covariance, show_noise):
    global current_noise, last_noise_params
    harmonic = amplitude * np.sin(2 * np.pi * frequency * t + phase)
    
    if last_noise_params[0] != noise_mean or last_noise_params[1] != noise_covariance:
        current_noise = np.random.normal(noise_mean, np.sqrt(noise_covariance), len(t))
        last_noise_params = [noise_mean, noise_covariance]
        
    return harmonic, (harmonic + current_noise if show_noise else harmonic)

Функція фільтрації

In [5]:
def apply_filter(data, cutoff, fs=100):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = signal.butter(4, normal_cutoff, btype='low', analog=False)
    return signal.filtfilt(b, a, data)

Створення вікна та осей

In [6]:
fig, ax = plt.subplots(figsize=(10, 8))
plt.subplots_adjust(left=0.1, bottom=0.35) # Залишаємо місце під слайдери

line_noisy, = ax.plot(t, np.zeros(len(t)), color='orange', alpha=0.5, label='Noisy')
line_orig, = ax.plot(t, np.zeros(len(t)), 'b--', linewidth=2, label='Original')
line_filt, = ax.plot(t, np.zeros(len(t)), 'r', linewidth=2, label='Filtered')

ax.set_ylim(-7, 7)
ax.legend(loc='upper right')
ax.grid(True)

Розміщення слайдерів гармоніки

In [7]:
ax_amp = plt.axes([0.15, 0.25, 0.65, 0.03])
ax_freq = plt.axes([0.15, 0.20, 0.65, 0.03])
ax_phase = plt.axes([0.15, 0.15, 0.65, 0.03])

s_amp = Slider(ax_amp, 'Amp', 0.1, 5.0, valinit=1.0)
s_freq = Slider(ax_freq, 'Freq', 0.1, 5.0, valinit=1.0)
s_phase = Slider(ax_phase, 'Phase', 0.0, 2*np.pi, valinit=0.0)

Розміщення слайдерів шуму та фільтра

In [8]:
ax_mean = plt.axes([0.15, 0.10, 0.25, 0.03])
ax_cov = plt.axes([0.55, 0.10, 0.25, 0.03])
ax_cut = plt.axes([0.15, 0.05, 0.65, 0.03])

s_mean = Slider(ax_mean, 'N-Mean', -1.0, 1.0, valinit=0.0)
s_cov = Slider(ax_cov, 'N-Cov', 0.01, 1.0, valinit=0.1)
s_cut = Slider(ax_cut, 'Filter', 0.1, 10.0, valinit=5.0)

Кнопка Reset та Checkbox

In [9]:
ax_reset = plt.axes([0.85, 0.20, 0.1, 0.05])
button = Button(ax_reset, 'Reset', color='red', hovercolor='0.975')

ax_check = plt.axes([0.85, 0.05, 0.1, 0.1])
check = CheckButtons(ax_check, ['Noise'], [True])

Основна логіка оновлення

In [10]:
def update(val):
    show_n = check.get_status()[0]
    h, _ = harmonic_with_noise(s_amp.val, s_freq.val, s_phase.val, 
                               s_mean.val, s_cov.val, True)
    
    full_noisy = h + current_noise
    f = apply_filter(full_noisy, s_cut.val)
    
    line_orig.set_ydata(h)
    line_filt.set_ydata(f)
    line_noisy.set_ydata(full_noisy)
    line_noisy.set_visible(show_n)
    
    fig.canvas.draw_idle()

s_amp.on_changed(update); s_freq.on_changed(update); s_phase.on_changed(update)
s_mean.on_changed(update); s_cov.on_changed(update); s_cut.on_changed(update)
check.on_clicked(update)

0

Скидання значень та запуск

In [11]:
def reset(event):
    s_amp.reset(); s_freq.reset(); s_phase.reset()
    s_mean.reset(); s_cov.reset(); s_cut.reset()
    update(None)

button.on_clicked(reset)
update(None)
plt.show()